In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
base_dir = Path(".").resolve()
data_dir = base_dir / "data"

In [3]:
with open(data_dir / "dist_rules.json") as file:
    data = json.load(file, object_hook=lambda obj: {**obj.pop("lvl", {}), **obj})

with open(data_dir / "metadata.json") as meta_file:
    metadata = json.load(meta_file)

In [4]:
rules = data.get("rules", [])

for rule in rules:
    rrg = rule.get("rrg")
    if rrg is None:
        rule["rrg"] = {}

    for key in list(rule.keys()):
        if key in metadata:
            rule[metadata[key]] = rule.pop(key)

    level_mapping = metadata["level_mapping"]
    for key, value in rule.items():
        if isinstance(value, dict):
            for k in list(value.keys()):
                if k in level_mapping:
                    value[level_mapping[k]] = value.pop(k)

df = pd.json_normalize(rules, sep="_")

In [5]:
df["client_list"] = df["client_list"].apply(lambda x: [] if x is None else x)
# df["provider_values"] = df["provider_values"].apply(lambda x: [] if x is None else x)

In [9]:
def count_rules(client_id, df):
    count = 0
    for rule in df["client_list"]:
        if str(client_id) in rule:
            count += 1
    return count


client_id = 11653
rule_count = count_rules(client_id, df)
print(f"Client {client_id} has {rule_count} applicable rules.")

Client 11653 has 335 applicable rules.
